In [1]:
%load_ext autoreload
import datetime as dt
import os
import sys
from itertools import product

import yaml

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, project_root)

In [2]:
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ipywidgets import interact
from scipy.signal import periodogram
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf

from src.data.download_yfinance import download_ticker_data

### Load data

In [3]:
# declare period to load into notebook 
# Note: (yfinance will have freshest data every morning CEST)
params = {
    'start_date': '2022-01-01',
    'end_date': dt.datetime.today()
}

In [4]:
# declare tickers + tickers sanity test
with open("../config/tickers_list.yaml", "r") as f:
    tickers = yaml.safe_load(f)["tickers"]

tickers[:5]

['AAPL', 'MSFT', 'GDX', 'IONQ', '7974.T']

In [5]:
intervals = ["1d", "1wk", "1mo"]
all_dfs = []

for ticker, interval in product(tickers, intervals):
    df_i = download_ticker_data(
        tickers=[ticker],
        start=params["start_date"],
        end=params["end_date"],
        interval=interval,
    )
    if df_i is not None and not df_i.empty:
        df_i["ticker"] = ticker
        df_i["interval"] = interval
        all_dfs.append(df_i)

df_all = pd.concat(all_dfs)

### Basic checks

In [6]:
print(f'Number of rows and columns: {df_all.shape[0], df_all.shape[1]}')

Number of rows and columns: (35036, 8)


In [7]:
print(f'data starts: {df_all.date.min()}')
print(f'data ends: end_date = {df_all.date.max()}')

data starts: 2022-01-01 00:00:00
data ends: end_date = 2025-07-09 00:00:00


### Plot options

In [8]:
ticker = widgets.Dropdown(options=sorted(tickers), description="Ticker")
interval = widgets.RadioButtons(options=["1d", "1wk", "1mo"])
seasonality_focus = widgets.RadioButtons(
    options=["generic", "yearly"], 
    description="Seasonality")

In [ ]:
# TODO: create fake ticker with perfect seasonality for comparison.
# it should be added to the tickers dataset

# Dashboards
## STL Seasonality Strength

Quantifies how much of a time series’ variation is explained by its seasonal component, as extracted using STL (Seasonal-Trend decomposition using Loess). It is calculated as:

$$S = \max\left(0,\ 1 - \frac{\operatorname{Var}(\text{remainder})}{\operatorname{Var}(\text{remainder} + \text{seasonal})} \right)$$


Values near 1 indicate strong seasonality; near 0 means weak or no seasonality.

In [9]:
@interact(ticker=ticker, interval=interval, seasonality=seasonality_focus)
def show_stl_strength(ticker, interval, seasonality):
    df = df_all[
        (df_all["ticker"] == ticker) & (df_all["interval"] == interval)
        ].copy()
    df["date"] = pd.to_datetime(df["date"])
    
    if df.empty:
        print("No data available.")
        return

    df.set_index("date", inplace=True)
    df = df.sort_index()
    series = df["close"].dropna()

    seasonal_periods = {
        "1d": {"generic": 252, "yearly": 365},
        "1wk": {"generic": 52, "yearly": 52},
        "1mo": {"generic": 12, "yearly": 12}
    }
    period = seasonal_periods[interval][seasonality]

    try:
        stl = STL(series, period=period, robust=True)
        result = stl.fit()
    except Exception as e:
        print(f"STL failed: {e}")
        return

    remainder = result.resid
    seasonal = result.seasonal
    strength = max(0, 1 - (np.var(remainder) / np.var(remainder + seasonal)))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=series.index, 
        y=series, 
        name="Close"))
    fig.add_trace(go.Scatter(
        x=series.index, 
        y=result.trend, 
        name="Trend"))
    fig.add_trace(go.Scatter(
        x=series.index, 
        y=seasonal, 
        name="Seasonal"))

    fig.update_layout(
        title= (
            f"{ticker} ({interval}) — STL Seasonality Strength"
            f"({seasonality}): {strength:.2f}"
        ),
        xaxis_title="Date",
        yaxis_title="Price",
        template="plotly_white"
    )
    fig.show()

interactive(children=(Dropdown(description='Ticker', options=('7974.T', 'AAPL', 'AVAV', 'BB', 'BMBL', 'CGW', '…

**STL Rule of Thumb:**
- Green seasonal line should have regular up-down waves, consistent amplitude and visible periodicity.
- It should also oscillate around zer and not overlap much with trend.
- Trend should capture log-term drift.
- Score domain is $> 0.6$: strong seasonality, $0.3 - 0.6$: moderate seasonality, $< 0.3$: negligible seasonality.


## ACF at Seasonal Lag

Refers to the value of the autocorrelation function at the lag corresponding to the seasonality period (e.g., lag = 12 for monthly data with yearly seasonality).

$$\text{ACF}_{s} = \rho(s)$$

where:
- $\text{ACF}_{s}$ is the autocorrelation at seasonal lag $s$
- $\rho(s)$ denotes the autocorrelation function evaluated at lag $s$

In [10]:
@interact(ticker=ticker, interval=interval)
def show_acf_seasonality(ticker, interval):
    df = df_all[(df_all["ticker"] == ticker) & (df_all["interval"] == interval)].copy()
    df["date"] = pd.to_datetime(df["date"])
    
    if df.empty:
        print("No data available.")
        return

    df.set_index("date", inplace=True)
    df = df.sort_index()
    series = df["close"].dropna()

    # seasonal lag definitions
    seasonal_lags = {
        "1d": 252,
        "1wk": 52,
        "1mo": 12
    }
    lag = seasonal_lags.get(interval, 12)

    # compute ACF
    max_lag = lag + 10
    acf_vals = acf(series, nlags=max_lag, fft=True)

    # value at seasonal lag
    seasonal_acf = acf_vals[lag]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=list(range(len(acf_vals))), 
        y=acf_vals, 
        name="ACF"))
    fig.add_trace(go.Scatter(
        x=[lag], 
        y=[seasonal_acf], 
        mode="markers+text", 
        text=[f"{seasonal_acf:.2f}"], 
        textposition="top center",
        marker=dict(color="red", size=10),
        name=f"ACF @ lag {lag}"
    ))

    fig.update_layout(
        title=f"{ticker} ({interval}) — ACF at Lag {lag}: {seasonal_acf:.2f}",
        xaxis_title="Lag",
        yaxis_title="ACF",
        template="plotly_white"
    )
    fig.show()

interactive(children=(Dropdown(description='Ticker', options=('7974.T', 'AAPL', 'AVAV', 'BB', 'BMBL', 'CGW', '…

**ACF Rule of Thumb**
- ACF at the expected lag is clearly elevated (local maxima good to have but not required).
- Spike stands out from general decay and monotonic decay with string persistence.
- Echo peaks at multiples of the lag are visible.
- Score domain is $> 0.5$: strong seasonality (albeit very rare), $0.3- 0.5$: moderate seasonality, $0.15-0.3$: weak, possibly noise induced, $< 0.15$: most probably not seasonal.

## Periodgram

The periodogram helps identify dominant periodicities in a time series by analyzing its spectral density. A high peak-to-background ratio suggests a strong recurring cycle (seasonality), while a flat spectrum indicates randomness.

$$\text{Peak-to-background ratio} = \frac{\max(P(f))}{\text{mean}(P(f))}$$

where:
- $P(f)$ is the power spectral density at frequency $fF$

In [11]:
@interact(ticker=ticker, interval=interval)
def show_periodogram(ticker, interval):
    df = df_all[(df_all["ticker"] == ticker) & (df_all["interval"] == interval)].copy()
    df["date"] = pd.to_datetime(df["date"])

    if df.empty:
        print("No data available.")
        return

    df.set_index("date", inplace=True)
    df = df.sort_index()

    series = df["close"].dropna()

    # periodogram
    frequencies, power = periodogram(series)
    peak_to_mean = power.max() / power.mean()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=frequencies, 
        y=power, 
        mode="lines", 
        name="Spectral Power"))

    title = (
        f"{ticker} ({interval}) — Periodogram "
        f"(Peak-to-Mean Ratio: {peak_to_mean:.2f})"
    )
    
    fig.update_layout(
        title= title,
        xaxis_title="Frequency",
        yaxis_title="Power",
        template="plotly_white"
    )

    fig.show()

interactive(children=(Dropdown(description='Ticker', options=('7974.T', 'AAPL', 'AVAV', 'BB', 'BMBL', 'CGW', '…

**Periodogram Rule of Thumb**
- Sharp, narrow peak at non-zero frequency
- High peak-to-mean ratio ($>20–30$) and peak is not at frequency $≈ 0$
- PMR is only meaningful when paired with frequency analysis. A high PMR at frequency $≈ 0$ is likely just trend.
- Domain values might be misleading in terms of seasonality: $1-5$: no dominant frequency, $5-15$: possibly weak seasonality or multiple low peaks, $15-30$: seasonality could be present, check alignment, $>30$: check if peak is not 0, if so, strong seasonal signal, $>100$: Check peak, possibly trend dominated. 

## Seasonal subseries (ANOVA)

This approach visualizes seasonality by grouping data into seasonal periods (e.g., months) and comparing their distributions. If the between-group variance is high compared to within-group variance, it suggests strong seasonality. This is conceptually similar to a one-way ANOVA test.

$$F = \frac{\text{Between-group variance}}{\text{Within-group variance}}$$

where:
- Flat or similar boxplots across months → low seasonality
- Distinct differences across months → strong seasonality

In [12]:
@interact(ticker=ticker, interval=interval)
def show_seasonal_subseries(ticker, interval):
    df = df_all[(df_all["ticker"] == ticker) & (df_all["interval"] == interval)].copy()
    df["date"] = pd.to_datetime(df["date"])

    if df.empty:
        print("No data available.")
        return

    df.set_index("date", inplace=True)
    df = df.sort_index()

    # grouping logic based on interval
    if interval == "1d":
        df["season"] = df.index.day
        season_label = "Month"
    elif interval == "1wk":
        df["season"] = df.index.isocalendar().week
        season_label = "Week"
    elif interval == "1mo":
        df["season"] = df.index.month
        season_label = "Month"
    else:
        print("Unsupported interval.")
        return

    fig = go.Figure()

    for s in sorted(df["season"].unique()):
        subset = df[df["season"] == s]["close"]
        fig.add_trace(go.Box(y=subset, name=str(s), boxmean=True))

    fig.update_layout(
        title=f"{ticker} ({interval}) — Seasonal Subseries by {season_label}",
        xaxis_title=season_label,
        yaxis_title="Close Price",
        template="plotly_white"
    )
    fig.show()

interactive(children=(Dropdown(description='Ticker', options=('7974.T', 'AAPL', 'AVAV', 'BB', 'BMBL', 'CGW', '…

**SSA Rule of Thumb**
- Clear, systematic differences across seasonal groups (e.g., months or weeks).
- Boxplots are well-separated in their medians, have relatively low overlap in interquartile ranges (IQRs)
- Outliers cluster differently per group
- Visual implication: Between-group variance ≫ within-group variance

## Quadratic fit

Tests whether a quadratic model can explain the seasonal pattern in average prices across time units (e.g., months, weeks, days in year).
This method fits a second-degree polynomial to the average close price across each seasonal unit. A good fit (high R2R2) suggests smooth seasonality.

$$y = ax^2 + bx + c$$

$$R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$$

where:
- $x$ is the seasonal unit
- $y$ is the average close price
- $a$, $b$, $c$ are the regression coefficients

In [13]:
@interact(ticker=ticker, interval=interval, seasonality=seasonality_focus)
def show_quadratic_fit(ticker, interval, seasonality):
    df = df_all[(df_all["ticker"] == ticker) & (df_all["interval"] == interval)].copy()
    df["date"] = pd.to_datetime(df["date"])

    if df.empty:
        print("No data available.")
        return

    df.set_index("date", inplace=True)
    df = df.sort_index()

    if seasonality == "monthly":
        df["seasonal_index"] = df.index.month
        x_label = "Month"
    elif seasonality == "weekly":
        df["seasonal_index"] = df.index.isocalendar().week
        x_label = "Week"
    elif seasonality == "yearly":
        df["seasonal_index"] = df.index.dayofyear
        x_label = "Day of Year"
    else:
        print("Invalid seasonality.")
        return

    grouped = df.groupby("seasonal_index")["close"].mean().reset_index()
    x = grouped["seasonal_index"]
    y = grouped["close"]

    if len(x) < 3:
        print("Not enough data to fit.")
        return

    coeffs = np.polyfit(x, y, 2)
    poly = np.poly1d(coeffs)
    y_fit = poly(x)

    r2 = 1 - np.sum((y - y_fit) ** 2) / np.sum((y - y.mean()) ** 2)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode="markers+lines", name="Avg Close"))
    fig.add_trace(go.Scatter(x=x, y=y_fit, mode="lines", name="Quadratic Fit"))

    fig.update_layout(
        title=f"{ticker} ({interval}) — Quadratic Fit by {seasonality} (R²={r2:.2f})",
        xaxis_title=x_label,
        yaxis_title="Average Close Price",
        template="plotly_white"
    )
    fig.show()

interactive(children=(Dropdown(description='Ticker', options=('7974.T', 'AAPL', 'AVAV', 'BB', 'BMBL', 'CGW', '…